In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv('injection_rates.csv')

# Onshore first, offshore second; sorted by mean within each group
df['_order'] = df['Typology'].map({'Onshore': 0, 'Offshore': 1})
df = df.sort_values(['_order', 'Saline Aquifer Mean (Mt/a)']).reset_index(drop=True)

basins   = df['Basin'].tolist()
means    = df['Saline Aquifer Mean (Mt/a)'].tolist()
mins     = df['Saline Aquifer Min (Mt/a)'].tolist()
maxs     = df['Saline Aquifer Max (Mt/a)'].tolist()
typology = df['Typology'].tolist()

n = len(basins)
x = np.arange(n)
tick_hw = 0.18   # half-width of horizontal cap ticks

fig, ax = plt.subplots(figsize=(20, 7))

for i in range(n):
    mn, me, mx = mins[i], means[i], maxs[i]

    # Vertical lines
    ax.plot([i, i], [me, mx], color='steelblue', linewidth=1.8, solid_capstyle='round')
    ax.plot([i, i], [mn, me], color='crimson',   linewidth=1.8, solid_capstyle='round')

    # Horizontal cap ticks
    for y_val, color in [(mx, 'steelblue'), (mn, 'crimson'), (me, 'dimgray')]:
        ax.plot([i - tick_hw, i + tick_hw], [y_val, y_val], color=color, linewidth=1.8)

    # Centre dot (red fill, blue edge)
    ax.scatter(i, me, color='crimson', edgecolors='steelblue',
               s=55, zorder=5, linewidths=1.8)

# Shade offshore basins lightly
for i, typ in enumerate(typology):
    if typ == 'Offshore':
        ax.axvspan(i - 0.5, i + 0.5, color='lightyellow', alpha=0.6, zorder=0)

# Vertical divider between onshore and offshore
n_onshore = typology.index('Offshore')
ax.axvline(n_onshore - 0.5, color='gray', linewidth=1, linestyle='--', alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(basins, rotation=45, ha='right', fontsize=8.5)
ax.set_xlim(-0.6, n - 0.3)
ax.set_ylabel('Annual CO₂ Injection Rate (Mt/a)', fontsize=12)
ax.set_title(
    'Annual CO₂ Injection Rates for Deep Saline Aquifers in China\n'
    'by Sedimentary Basin',
    fontsize=13
)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='steelblue', label='Max'),
    mpatches.Patch(facecolor='dimgray',   label='Mean'),
    mpatches.Patch(facecolor='crimson',   label='Min'),
    mpatches.Patch(facecolor='lightyellow', edgecolor='gray',
                   alpha=0.8, label='Offshore'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.8)

plt.tight_layout()
plt.savefig('injection_rates_saline_aquifers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: injection_rates_saline_aquifers.png')

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

# ── CONFIG ────────────────────────────────────────────────────────────────────
VALUE_COL    = 'Saline Aquifer Mean (Mt/a)'   # swap in Min / Max / SD as needed
PIE_SCALE    = 2.0                             # scale all circles up/down uniformly
LEGEND_SIZES = [10, 50, 100]                  # Mt/a reference circles in the legend

# ── LOAD MAP ──────────────────────────────────────────────────────────────────
GEOJSON = '/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/gadm36_CHN_1.json'
gdf = gpd.read_file(GEOJSON)
gdf["NAME_1"] = gdf["NAME_1"].replace({
    "Nei Mongol":     "Innermongolia",
    "Ningxia Hui":    "Ningxia",
    "Xinjiang Uygur": "Xinjiang",
    "Xizang":         "Tibet",
})

# ── LOAD BASIN CENTROIDS AND INJECTION RATES ──────────────────────────────────
centroids = pd.read_csv('../usgs_source/basin_centroids_shapefile.csv', index_col=0)
rates     = pd.read_csv('injection_rates.csv')

# Harmonise basin names between the two CSVs
rates['Basin'] = rates['Basin'].replace({
    'Tuepan-Hami Basin': 'Turpan-Hami Basin',
    'Ejinjina Basin':    'Yingen-Ejina Basin',
})

merged = centroids.merge(rates[['Basin', VALUE_COL]], left_on='basin', right_on='Basin', how='left')

# ── PROJECT TO WEB-MERCATOR ───────────────────────────────────────────────────
gdf_proj = gdf.to_crs(epsg=3857)

basin_gdf = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged['lon'], merged['lat']),
    crs='EPSG:4326'
).to_crs(epsg=3857)

basin_gdf['cx'] = basin_gdf.geometry.x
basin_gdf['cy'] = basin_gdf.geometry.y

# ── SIZE CIRCLES ──────────────────────────────────────────────────────────────
max_val = basin_gdf[VALUE_COL].max()
basin_gdf['radius'] = (basin_gdf[VALUE_COL] / max_val) * PIE_SCALE * 80_000

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_axis_off()

gdf_proj.plot(ax=ax, color='#F4F4F4', edgecolor='black', linewidth=0.7, zorder=0)
gdf_proj.boundary.plot(ax=ax, linewidth=0.7, color='black', zorder=1)

onshore_color  = 'crimson'
offshore_color = '#08306b'   # dark navy blue

for _, row in basin_gdf.iterrows():
    val = row[VALUE_COL]
    if pd.isna(val) or val == 0:
        continue
    color = onshore_color if row['type'] == 'onshore' else offshore_color
    circle = Circle(
        (row['cx'], row['cy']), row['radius'],
        facecolor=color, edgecolor='white', linewidth=0.8,
        alpha=0.75, zorder=10
    )
    ax.add_patch(circle)

# ── SIZE LEGEND ───────────────────────────────────────────────────────────────
xlim, ylim = ax.get_xlim(), ax.get_ylim()
plot_w, plot_h = xlim[1] - xlim[0], ylim[1] - ylim[0]
pad_x, pad_y   = plot_w * 0.02, plot_h * 0.02
legend_radii   = [(s / max_val) * PIE_SCALE * 80_000 for s in LEGEND_SIZES]
max_r          = max(legend_radii)
legend_cx      = xlim[0] + pad_x + max_r

y = ylim[0] + pad_y
for size, radius in sorted(zip(LEGEND_SIZES, legend_radii)):
    cy_leg = y + radius
    ax.add_patch(Circle(
        (legend_cx, cy_leg), radius,
        facecolor='white', edgecolor='#555555', linewidth=1.2, zorder=20
    ))
    ax.text(
        legend_cx + max_r + pad_x * 0.6, cy_leg,
        f'{size} Mt/a', va='center', ha='left', fontsize=9, zorder=21
    )
    y += radius * 2 + pad_y * 0.4

# ── COLOUR LEGEND ─────────────────────────────────────────────────────────────
ax.legend(
    handles=[
        Line2D([0],[0], marker='o', color='w', markerfacecolor=onshore_color,
               markeredgecolor='white', markersize=10, label='Onshore'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor=offshore_color,
               markeredgecolor='white', markersize=10, label='Offshore'),
    ],
    loc='lower right', fontsize=10, framealpha=0.8
)

col_label = VALUE_COL.replace(' (Mt/a)', '')
plt.title(f'Annual CO₂ Injection Rate — Deep Saline Aquifers\n({col_label}, Mt/a)', fontsize=14)
plt.tight_layout()
plt.savefig(f'injection_rates_map_{col_label.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
plt.show()